# Model Serving with FastAPI

## Learning Objectives
- Understand REST API fundamentals for ML model serving
- Build production-ready APIs with FastAPI
- Implement input validation with Pydantic
- Handle model loading and inference
- Add health checks and documentation

## Why Model Serving Matters

A trained model is only valuable when it can make predictions in production. Model serving involves:

1. **API Design**: RESTful endpoints for predictions
2. **Input Validation**: Ensure data quality at inference time
3. **Performance**: Low latency, high throughput
4. **Reliability**: Error handling, health checks
5. **Documentation**: Auto-generated API docs

## 1. FastAPI Fundamentals

FastAPI is a modern, high-performance web framework for building APIs with Python.

### Key Features:
- **Fast**: Very high performance (on par with NodeJS and Go)
- **Type Hints**: Full Python type hints support
- **Auto Documentation**: Swagger UI and ReDoc
- **Validation**: Pydantic-based request/response validation
- **Async Support**: Native async/await support

In [ ]:
# Install FastAPI and dependencies (run in terminal)
# pip install fastapi[standard] uvicorn pydantic

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from typing import List, Optional, Dict, Any, Union
from datetime import datetime
import json

print("Imports successful!")

## 2. Building a Simple ML Model for Serving

First, let's train a simple model that we'll serve via FastAPI.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Generate sample data
np.random.seed(42)
X, y = make_classification(
    n_samples=1000,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    n_classes=2,
    random_state=42
)

# Feature names for documentation
feature_names = ['feature_1', 'feature_2', 'feature_3', 'feature_4']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create pipeline
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Save the model for serving
model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)

model_path = model_dir / 'classifier_model.joblib'
joblib.dump(model_pipeline, model_path)

# Save model metadata
model_metadata = {
    'model_name': 'binary_classifier',
    'model_version': '1.0.0',
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'classes': [0, 1],
    'training_accuracy': float(accuracy),
    'created_at': datetime.now().isoformat(),
    'framework': 'scikit-learn',
    'framework_version': '1.3.0'
}

metadata_path = model_dir / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"Model saved to: {model_path}")
print(f"Metadata saved to: {metadata_path}")

## 3. Pydantic Models for Request/Response Validation

Pydantic models provide automatic validation and documentation for API inputs/outputs.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum

# Request Models
class PredictionInput(BaseModel):
    """Input schema for single prediction."""
    feature_1: float = Field(..., description="First feature value")
    feature_2: float = Field(..., description="Second feature value")
    feature_3: float = Field(..., description="Third feature value")
    feature_4: float = Field(..., description="Fourth feature value")
    
    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "feature_1": 0.5,
                    "feature_2": -1.2,
                    "feature_3": 0.8,
                    "feature_4": 0.3
                }
            ]
        }
    }


class BatchPredictionInput(BaseModel):
    """Input schema for batch predictions."""
    instances: List[PredictionInput] = Field(
        ..., 
        description="List of prediction inputs",
        min_length=1,
        max_length=1000
    )


# Response Models
class PredictionOutput(BaseModel):
    """Output schema for single prediction."""
    prediction: int = Field(..., description="Predicted class (0 or 1)")
    probability: float = Field(..., description="Probability of predicted class")
    probabilities: List[float] = Field(..., description="Probabilities for all classes")
    model_version: str = Field(..., description="Model version used")


class BatchPredictionOutput(BaseModel):
    """Output schema for batch predictions."""
    predictions: List[PredictionOutput]
    count: int
    processing_time_ms: float


class HealthStatus(str, Enum):
    HEALTHY = "healthy"
    DEGRADED = "degraded"
    UNHEALTHY = "unhealthy"


class HealthResponse(BaseModel):
    """Health check response."""
    status: HealthStatus
    model_loaded: bool
    model_version: str
    uptime_seconds: float
    timestamp: str


# Test validation
test_input = PredictionInput(feature_1=0.5, feature_2=-1.2, feature_3=0.8, feature_4=0.3)
print(f"Valid input: {test_input}")
print(f"As dict: {test_input.model_dump()}")

## 4. Model Manager Class

A dedicated class to handle model loading, caching, and inference.

In [ ]:
import time
from threading import Lock

class ModelManager:
    """Manages model loading and inference."""
    
    def __init__(self, model_path: str, metadata_path: str):
        self.model_path = Path(model_path)
        self.metadata_path = Path(metadata_path)
        self.model = None
        self.metadata = None
        self._lock = Lock()
        self.start_time = time.time()
        self._load_model()
    
    def _load_model(self):
        """Load model and metadata from disk."""
        with self._lock:
            self.model = joblib.load(self.model_path)
            with open(self.metadata_path, 'r') as f:
                self.metadata = json.load(f)
        print(f"Model loaded: {self.metadata['model_name']} v{self.metadata['model_version']}")
    
    def reload_model(self):
        """Hot-reload model without downtime."""
        self._load_model()
    
    def predict(self, features: np.ndarray) -> Dict[str, Any]:
        """Make prediction with probability scores."""
        if self.model is None:
            raise RuntimeError("Model not loaded")
        
        prediction = self.model.predict(features)[0]
        probabilities = self.model.predict_proba(features)[0].tolist()
        
        return {
            'prediction': int(prediction),
            'probability': float(max(probabilities)),
            'probabilities': probabilities,
            'model_version': self.metadata['model_version']
        }
    
    def predict_batch(self, features: np.ndarray) -> List[Dict[str, Any]]:
        """Make batch predictions."""
        if self.model is None:
            raise RuntimeError("Model not loaded")
        
        predictions = self.model.predict(features)
        probabilities = self.model.predict_proba(features)
        
        results = []
        for i, (pred, probs) in enumerate(zip(predictions, probabilities)):
            results.append({
                'prediction': int(pred),
                'probability': float(max(probs)),
                'probabilities': probs.tolist(),
                'model_version': self.metadata['model_version']
            })
        
        return results
    
    def get_health(self) -> Dict[str, Any]:
        """Get health status."""
        is_loaded = self.model is not None
        status = 'healthy' if is_loaded else 'unhealthy'
        
        return {
            'status': status,
            'model_loaded': is_loaded,
            'model_version': self.metadata.get('model_version', 'unknown') if self.metadata else 'unknown',
            'uptime_seconds': time.time() - self.start_time,
            'timestamp': datetime.now().isoformat()
        }


# Test the model manager
manager = ModelManager(
    model_path='../models/classifier_model.joblib',
    metadata_path='../models/model_metadata.json'
)

# Test prediction
test_features = np.array([[0.5, -1.2, 0.8, 0.3]])
result = manager.predict(test_features)
print(f"\nPrediction result: {result}")

# Test health check
health = manager.get_health()
print(f"\nHealth status: {health}")

## 5. Complete FastAPI Application

Here's a complete FastAPI application for model serving. Save this as `app.py`:

In [ ]:
# FastAPI Application Code (save as app.py)
fastapi_app_code = '''
"""
ML Model Serving API with FastAPI

Run with: uvicorn app:app --reload --port 8000
Docs at: http://localhost:8000/docs
"""

from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from datetime import datetime
from enum import Enum
import numpy as np
import joblib
import json
import time
import logging
from pathlib import Path
from contextlib import asynccontextmanager

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ============== Pydantic Models ==============

class PredictionInput(BaseModel):
    """Input schema for single prediction."""
    feature_1: float = Field(..., description="First feature value")
    feature_2: float = Field(..., description="Second feature value")
    feature_3: float = Field(..., description="Third feature value")
    feature_4: float = Field(..., description="Fourth feature value")


class BatchPredictionInput(BaseModel):
    """Input schema for batch predictions."""
    instances: List[PredictionInput] = Field(
        ..., min_length=1, max_length=1000,
        description="List of prediction inputs"
    )


class PredictionOutput(BaseModel):
    """Output schema for prediction."""
    prediction: int
    probability: float
    probabilities: List[float]
    model_version: str


class BatchPredictionOutput(BaseModel):
    """Output schema for batch predictions."""
    predictions: List[PredictionOutput]
    count: int
    processing_time_ms: float


class HealthStatus(str, Enum):
    HEALTHY = "healthy"
    DEGRADED = "degraded"
    UNHEALTHY = "unhealthy"


class HealthResponse(BaseModel):
    """Health check response."""
    status: HealthStatus
    model_loaded: bool
    model_version: str
    uptime_seconds: float
    timestamp: str


class ModelInfo(BaseModel):
    """Model metadata."""
    model_name: str
    model_version: str
    feature_names: List[str]
    n_features: int
    classes: List[int]
    training_accuracy: float


# ============== Model Manager ==============

class ModelManager:
    """Manages ML model lifecycle."""
    
    def __init__(self):
        self.model = None
        self.metadata = None
        self.start_time = time.time()
    
    def load(self, model_path: str, metadata_path: str):
        """Load model from disk."""
        self.model = joblib.load(model_path)
        with open(metadata_path, "r") as f:
            self.metadata = json.load(f)
        logger.info(f"Loaded model: {self.metadata[\'model_name\']} v{self.metadata[\'model_version\']}")
    
    def predict(self, features: np.ndarray) -> Dict[str, Any]:
        """Make single prediction."""
        pred = self.model.predict(features)[0]
        probs = self.model.predict_proba(features)[0]
        return {
            "prediction": int(pred),
            "probability": float(max(probs)),
            "probabilities": probs.tolist(),
            "model_version": self.metadata["model_version"]
        }
    
    def predict_batch(self, features: np.ndarray) -> List[Dict]:
        """Make batch predictions."""
        preds = self.model.predict(features)
        probs = self.model.predict_proba(features)
        return [
            {
                "prediction": int(p),
                "probability": float(max(pr)),
                "probabilities": pr.tolist(),
                "model_version": self.metadata["model_version"]
            }
            for p, pr in zip(preds, probs)
        ]


# Global model manager
model_manager = ModelManager()


# ============== Lifespan Handler ==============

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Handle startup and shutdown."""
    # Startup: Load model
    model_path = Path("models/classifier_model.joblib")
    metadata_path = Path("models/model_metadata.json")
    
    if model_path.exists() and metadata_path.exists():
        model_manager.load(str(model_path), str(metadata_path))
    else:
        logger.warning("Model files not found!")
    
    yield  # Application runs here
    
    # Shutdown: Cleanup
    logger.info("Shutting down...")


# ============== FastAPI App ==============

app = FastAPI(
    title="ML Model Serving API",
    description="Production-ready API for serving ML predictions",
    version="1.0.0",
    lifespan=lifespan
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ============== Endpoints ==============

@app.get("/")
async def root():
    """Root endpoint."""
    return {"message": "ML Model Serving API", "docs": "/docs"}


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint for load balancers."""
    is_loaded = model_manager.model is not None
    return HealthResponse(
        status=HealthStatus.HEALTHY if is_loaded else HealthStatus.UNHEALTHY,
        model_loaded=is_loaded,
        model_version=model_manager.metadata.get("model_version", "unknown") if model_manager.metadata else "unknown",
        uptime_seconds=time.time() - model_manager.start_time,
        timestamp=datetime.now().isoformat()
    )


@app.get("/model/info", response_model=ModelInfo)
async def model_info():
    """Get model metadata."""
    if model_manager.metadata is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    return ModelInfo(**model_manager.metadata)


@app.post("/predict", response_model=PredictionOutput)
async def predict(input_data: PredictionInput):
    """Make a single prediction."""
    if model_manager.model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    try:
        features = np.array([[
            input_data.feature_1,
            input_data.feature_2,
            input_data.feature_3,
            input_data.feature_4
        ]])
        result = model_manager.predict(features)
        return PredictionOutput(**result)
    except Exception as e:
        logger.error(f"Prediction error: {e}")
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch", response_model=BatchPredictionOutput)
async def predict_batch(input_data: BatchPredictionInput):
    """Make batch predictions."""
    if model_manager.model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    try:
        start_time = time.time()
        
        features = np.array([
            [inst.feature_1, inst.feature_2, inst.feature_3, inst.feature_4]
            for inst in input_data.instances
        ])
        
        results = model_manager.predict_batch(features)
        processing_time = (time.time() - start_time) * 1000
        
        return BatchPredictionOutput(
            predictions=[PredictionOutput(**r) for r in results],
            count=len(results),
            processing_time_ms=processing_time
        )
    except Exception as e:
        logger.error(f"Batch prediction error: {e}")
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/model/reload")
async def reload_model():
    """Hot-reload the model."""
    try:
        model_manager.load(
            "models/classifier_model.joblib",
            "models/model_metadata.json"
        )
        return {"message": "Model reloaded successfully"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Reload failed: {e}")


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Save the FastAPI app
app_path = Path('../models/app.py')
with open(app_path, 'w') as f:
    f.write(fastapi_app_code)

print(f"FastAPI app saved to: {app_path}")
print("\nTo run the server:")
print("  cd models")
print("  uvicorn app:app --reload --port 8000")
print("\nAPI docs will be available at:")
print("  http://localhost:8000/docs (Swagger UI)")
print("  http://localhost:8000/redoc (ReDoc)")

## 6. Testing the API

FastAPI makes testing easy with the built-in TestClient.

In [ ]:
# Simulating API calls (without running the server)
import requests
from unittest.mock import Mock, patch

def simulate_prediction_call(input_data: dict) -> dict:
    """Simulate what the API would return."""
    features = np.array([[
        input_data['feature_1'],
        input_data['feature_2'],
        input_data['feature_3'],
        input_data['feature_4']
    ]])
    
    return manager.predict(features)

# Test single prediction
test_input = {
    'feature_1': 0.5,
    'feature_2': -1.2,
    'feature_3': 0.8,
    'feature_4': 0.3
}

result = simulate_prediction_call(test_input)
print("Single Prediction Test:")
print(f"  Input: {test_input}")
print(f"  Output: {result}")

# Test batch prediction
batch_inputs = [
    {'feature_1': 0.5, 'feature_2': -1.2, 'feature_3': 0.8, 'feature_4': 0.3},
    {'feature_1': -0.5, 'feature_2': 1.2, 'feature_3': -0.8, 'feature_4': -0.3},
    {'feature_1': 1.0, 'feature_2': 0.0, 'feature_3': 1.0, 'feature_4': 0.0},
]

features = np.array([[d['feature_1'], d['feature_2'], d['feature_3'], d['feature_4']] 
                     for d in batch_inputs])
batch_results = manager.predict_batch(features)

print("\nBatch Prediction Test:")
for i, (inp, res) in enumerate(zip(batch_inputs, batch_results)):
    print(f"  Instance {i+1}: pred={res['prediction']}, prob={res['probability']:.3f}")

## 7. API Client Implementation

A reusable client for consuming the prediction API.

In [ ]:
from typing import Union
import time

class PredictionClient:
    """
    Client for interacting with the ML Prediction API.
    
    Example:
        client = PredictionClient('http://localhost:8000')
        result = client.predict([0.5, -1.2, 0.8, 0.3])
    """
    
    def __init__(self, base_url: str, timeout: int = 30):
        self.base_url = base_url.rstrip('/')
        self.timeout = timeout
        self.session = None  # Would use requests.Session() in production
    
    def health_check(self) -> dict:
        """Check API health."""
        # In production: return self.session.get(f"{self.base_url}/health").json()
        return manager.get_health()
    
    def get_model_info(self) -> dict:
        """Get model metadata."""
        # In production: return self.session.get(f"{self.base_url}/model/info").json()
        return manager.metadata
    
    def predict(self, features: Union[list, np.ndarray]) -> dict:
        """Make single prediction."""
        if isinstance(features, np.ndarray):
            features = features.flatten().tolist()
        
        # In production: POST to /predict
        return manager.predict(np.array([features]))
    
    def predict_batch(self, instances: list) -> dict:
        """Make batch predictions."""
        start_time = time.time()
        
        features = np.array(instances)
        results = manager.predict_batch(features)
        
        processing_time = (time.time() - start_time) * 1000
        
        return {
            'predictions': results,
            'count': len(results),
            'processing_time_ms': processing_time
        }


# Demo the client
client = PredictionClient('http://localhost:8000')

print("API Health:")
print(f"  {client.health_check()}")

print("\nModel Info:")
info = client.get_model_info()
print(f"  Name: {info['model_name']}")
print(f"  Version: {info['model_version']}")
print(f"  Features: {info['feature_names']}")

print("\nSingle Prediction:")
result = client.predict([0.5, -1.2, 0.8, 0.3])
print(f"  Prediction: {result['prediction']}")
print(f"  Confidence: {result['probability']:.2%}")

## 8. Performance Considerations

Key considerations for production model serving.

In [ ]:
import time

def benchmark_inference(n_samples: int = 1000, batch_size: int = 32):
    """
    Benchmark model inference performance.
    
    In production, you want:
    - p50 latency < 50ms for real-time serving
    - p99 latency < 200ms
    - Throughput > 100 requests/second
    """
    # Generate random test data
    X_test = np.random.randn(n_samples, 4)
    
    # Benchmark single predictions
    latencies = []
    for i in range(min(100, n_samples)):
        start = time.perf_counter()
        _ = manager.predict(X_test[i:i+1])
        latencies.append((time.perf_counter() - start) * 1000)
    
    print("=== Single Prediction Latency ===")
    print(f"  p50: {np.percentile(latencies, 50):.2f} ms")
    print(f"  p90: {np.percentile(latencies, 90):.2f} ms")
    print(f"  p99: {np.percentile(latencies, 99):.2f} ms")
    print(f"  Mean: {np.mean(latencies):.2f} ms")
    
    # Benchmark batch predictions
    batch_latencies = []
    n_batches = n_samples // batch_size
    
    for i in range(n_batches):
        batch = X_test[i*batch_size:(i+1)*batch_size]
        start = time.perf_counter()
        _ = manager.predict_batch(batch)
        batch_latencies.append((time.perf_counter() - start) * 1000)
    
    print(f"\n=== Batch Prediction (batch_size={batch_size}) ===")
    print(f"  p50: {np.percentile(batch_latencies, 50):.2f} ms")
    print(f"  p90: {np.percentile(batch_latencies, 90):.2f} ms")
    print(f"  Mean: {np.mean(batch_latencies):.2f} ms")
    
    # Calculate throughput
    total_time = sum(batch_latencies) / 1000  # Convert to seconds
    throughput = (n_batches * batch_size) / total_time
    print(f"  Throughput: {throughput:.0f} predictions/second")


benchmark_inference()

## 9. Best Practices Summary

### API Design
1. **Use Pydantic models** for request/response validation
2. **Version your API** (e.g., `/v1/predict`)
3. **Include model version** in responses for traceability
4. **Implement health checks** for load balancers

### Performance
1. **Load model once** at startup (not per-request)
2. **Use batch endpoints** for high throughput
3. **Add caching** for repeated predictions
4. **Profile and optimize** bottlenecks

### Reliability
1. **Graceful error handling** with proper HTTP status codes
2. **Request timeouts** to prevent hanging
3. **Hot model reload** without downtime
4. **Logging** for debugging and monitoring

### Security
1. **Input validation** to prevent malicious payloads
2. **Rate limiting** to prevent abuse
3. **Authentication** for sensitive models
4. **HTTPS** in production

In [ ]:
# Summary of key files created
print("=" * 60)
print("MODEL SERVING SETUP COMPLETE")
print("=" * 60)
print("\nFiles created:")
print(f"  📦 Model: models/classifier_model.joblib")
print(f"  📋 Metadata: models/model_metadata.json")
print(f"  🚀 API: models/app.py")
print("\nTo start the server:")
print("  cd models")
print("  pip install fastapi[standard] uvicorn")
print("  uvicorn app:app --reload --port 8000")
print("\nEndpoints:")
print("  GET  /health        - Health check")
print("  GET  /model/info    - Model metadata")
print("  POST /predict       - Single prediction")
print("  POST /predict/batch - Batch predictions")
print("  POST /model/reload  - Hot reload model")
print("\nDocumentation:")
print("  http://localhost:8000/docs   (Swagger UI)")
print("  http://localhost:8000/redoc  (ReDoc)")